# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
**Profesor**: Pablo Camarillo Ramirez

## Carlos Muñiz Lara


In [1]:
from spark_utils import SparkUtils
su = SparkUtils()


In [2]:
agency_columns = [("agency_id", "int"), ("agency_info", "string")]
brand_columns = [("brand_id", "int"), ("brand_info", "string")]
car_columns = [("car_id", "int"), ("car_info", "string")]
customer_columns = [("customer_id", "int"), ("customer_info", "string")]
rental_columns = [("rental_id", "int"), ("rental_info", "string")]


In [ ]:
agency_schema = SparkUtils.generate_schema(agency_columns)
brand_schema = SparkUtils.generate_schema(brand_columns)
car_schema = SparkUtils.generate_schema(car_columns)
customer_schema = SparkUtils.generate_schema(customer_columns)
rental_schema = SparkUtils.generate_schema(rental_columns)


In [6]:
agency_df = su._spark.read.option("header", "true").schema(agency_schema).csv("/opt/spark/work-dir/data/car_service/agencies")
brand_df = su._spark.read.option("header", "true").schema(brand_schema).csv("/opt/spark/work-dir/data/car_service/brands")
cars_df = su._spark.read.option("header", "true").schema(car_schema).csv("/opt/spark/work-dir/data/car_service/cars")
customer_df = su._spark.read.option("header", "true").schema(customer_schema).csv("/opt/spark/work-dir/data/car_service/customers")
rental_df = su._spark.read.option("header", "true").schema(rental_schema).csv("/opt/spark/work-dir/data/car_service/rentals")

In [ ]:
from pyspark.sql.functions import get_json_object
agency_df = agency_df.withColumn("agency_name", get_json_object(agency_df.agency_info,"$.agency_name"))
agency_df = agency_df.withColumn("city", get_json_object(agency_df.agency_info,"$.city"))


In [ ]:
brand_df = brand_df.withColumn("brand_name", get_json_object(brand_df.brand_info,"$.brand_name"))
brand_df = brand_df.withColumn("country", get_json_object(brand_df.brand_info,"$.country"))


In [ ]:
cars_df = cars_df.withColumn("car_name", get_json_object(cars_df.car_info,"$.car_name"))
cars_df = cars_df.withColumn("brand_id", get_json_object(cars_df.car_info,"$.brand_id").cast("int"))
cars_df = cars_df.withColumn("price_per_day", get_json_object(cars_df.car_info,"$.price_per_day").cast("int"))

In [ ]:
customer_df = customer_df.withColumn("customer_name", get_json_object(customer_df.customer_info,"$.customer_name"))
customer_df = customer_df.withColumn("age", get_json_object(customer_df.customer_info,"$.age").cast("int"))
customer_df = customer_df.withColumn("city", get_json_object(customer_df.customer_info,"$.city"))

In [ ]:
rental_df = rental_df.withColumn("customer_id", get_json_object(rental_df.rental_info,"$.customer_id").cast("int"))
rental_df = rental_df.withColumn("car_id", get_json_object(rental_df.rental_info,"$.car_id").cast("int"))
rental_df = rental_df.withColumn("agency_id", get_json_object(rental_df.rental_info,"$.agency_id").cast("int"))

In [ ]:
final_schema = [("rental_id","int"), ("car_name","string"), ("agency_name", "string"),("customer_name", "string")]

In [ ]:
final_df = SparkUtils.generate_schema(final_schema)


In [ ]:
final_df = rental_df \
.join(cars_df, rental_df["car_id"]== cars_df["car_id"], "inner") \
.join(customer_df, rental_df["customer_id"]== customer_df["customer_id"], "inner") \
.join(agency_df, rental_df["agency_id"]== agency_df["agency_id"], "inner") \
.select("rental_id","car_name", "customer_name", "agency_name")
final_df.show()

+---------+--------------------+---------------+-------------+
|rental_id|            car_name|  customer_name|  agency_name|
+---------+--------------------+---------------+-------------+
|    11891|Wallace-Carlson M...| Margaret Jones|  NYC Rentals|
|    11892|Grimes-Green Model 8|Albert Williams|LA Car Rental|
|    11893|Stewart-Allen Mod...|  Caleb Fleming|      SF Cars|
|    11894|  Campos PLC Model 4|  Andrew Butler|  NYC Rentals|
|    11895|  Wagner LLC Model 1|  Kristin Potts|      SF Cars|
|    11896|Jones, Jefferson ...|   Jeremy Parks|LA Car Rental|
|    11897|Lopez and Sons Mo...|    Terry Wells| Zapopan Auto|
|    11898| Salazar Ltd Model 8|  Marc Williams|      SF Cars|
|    11899|Villanueva PLC Mo...| Danny Williams|LA Car Rental|
|    11900|Faulkner-Howard M...| Eric Owens PhD|      SF Cars|
|    11901|Faulkner-Howard M...|    Laura Perry|  NYC Rentals|
|    11902|Faulkner-Howard M...|     Paul Brown|  NYC Rentals|
|    11903|Atkinson Ltd Mode...|Alexa Hernandez| Zapopa

In [ ]:
su._spark.stop()